# 3. Imputation of Missing Data 

In line with Step 3 of the OECD Handbook, this section addresses missing values to ensure a complete dataset before index construction. Leaving missing entries unresolved would distort comparisons and reduce the reliability of the final composite index.

### 3.1.1. Core Methodology:
* **Latest Value Selection:** To reduce reporting lag issues, the 2016-2026 timeline is converted into a cross-sectional dataset. For each country-indicator pair, the pipeline selects the most recent available value by scanning backward from 2026.

* **Group Average Imputation:** Some countries contain fully missing indicators across the entire timeline, particularly for advanced financial market variables. In these cases, missing values are estimated using peer-group averages.

* **Income Grouping:** Instead of applying a global mean, missing values are filled using the average of the country’s World Bank income group (High, Low, Lower-Middle, or Upper-Middle Income). This preserves regional and economic comparability while reducing imputation bias.

### 3.1.2. Post-Imputation Quality Diagnostics:
* **Impact Assessment:** The percentage of imputed values is calculated for each variable to evaluate overall dataset reliability after imputation.

* **Outlier Profile Review:** An Interquartile Range (IQR) analysis is applied to identify extreme values before normalization and index aggregation.

In [28]:
import os
import numpy as np
import pandas as pd

filtered_data_path = "../data/worldbank_filtered_data.csv"
metadata_path = "../data/worldbank_metadata.csv"

In [29]:
if not os.path.exists(filtered_data_path):
    print(f"'{filtered_data_path}' not found!")

else:
    df_filtered_load = pd.read_csv(filtered_data_path)

    # Create a latest-value snapshot from the timeline dataset
    df_compaction = df_filtered_load.copy()

    year_cols = sorted(
        [col for col in df_compaction.columns if col.isdigit()],
        reverse=True
    )

    # Select the most recent available value for each row
    def get_latest_value(row):
        for year in year_cols:
            if pd.notnull(row[year]):
                return row[year]
        return np.nan

    df_compaction['Latest_Value'] = df_compaction.apply(get_latest_value, axis=1)

    # Convert the dataset into a country-level matrix
    df_snapshot = df_compaction.pivot(
        index=['Country Name', 'Country Code'],
        columns='Variable',
        values='Latest_Value'
    ).reset_index()

In [30]:
# Merge official World Bank income group classifications
try:
    df_meta = pd.read_csv(metadata_path, encoding='latin1')

    df_meta = df_meta[['Code', 'Indicator Name']].rename(columns={
        'Code': 'Country Code',
        'Indicator Name': 'Income Group'
    })

    df_snapshot = pd.merge(
        df_snapshot,
        df_meta,
        on='Country Code',
        how='left'
    )

    print("Income group data merged successfully.")

except Exception as e:
    print(f"Metadata merge skipped: {e}")
    df_snapshot['Income Group'] = 'Middle income'

# Fill any remaining missing income groups
df_snapshot['Income Group'] = df_snapshot['Income Group'].fillna('Middle income')

Income group data merged successfully.


In [31]:
# Apply income-group imputation
df_before_imp = df_snapshot.copy()

numeric_cols = [
    col for col in df_snapshot.columns
    if col not in ['Country Name', 'Country Code', 'Income Group']
]

for col in numeric_cols:

    # Calculate average values within each income group
    group_means = df_snapshot.groupby('Income Group')[col].transform('mean')

    # Fill missing values using income-group averages
    df_snapshot[col] = df_snapshot[col].fillna(group_means)

    # Apply global mean if missing values remain
    if df_snapshot[col].isnull().sum() > 0:
        df_snapshot[col] = df_snapshot[col].fillna(
            df_snapshot[col].mean()
        )

print(
    f"Imputation completed. "
    f"Remaining missing values: "
    f"{df_snapshot[numeric_cols].isnull().sum().sum()}"
)


Imputation completed. Remaining missing values: 0


In [32]:
# Generate imputation and outlier analysis
print("\nGenerating data quality analysis")

# Create a dataframe to store outlier flags
df_outliers_log = df_snapshot[['Country Name', 'Country Code']].copy()

print("\n============================= DATA QUALITY REPORT =============================")
print(
    f"{'Variable Name'.ljust(30)} | "
    f"{'Imputation Rate (%)'.rjust(18)} | "
    f"{'Detected Outliers'.rjust(18)}"
)

print("-" * 74)

for col in numeric_cols:

    # Calculate percentage of imputed values
    missing_count = df_before_imp[col].isnull().sum()
    pct_imputed = (missing_count / len(df_snapshot)) * 100
    pct_str = f"{round(pct_imputed, 1)}%"

    # Calculate standard IQR outlier boundaries
    q1 = df_snapshot[col].quantile(0.25)
    q3 = df_snapshot[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify values outside the IQR bounds
    is_outlier = (
        (df_snapshot[col] < lower_bound) |
        (df_snapshot[col] > upper_bound)
    )

    num_outliers = is_outlier.sum()

    # Store outlier flags for each variable
    df_outliers_log[f"{col}_is_outlier"] = is_outlier

    print(
        f"{col.ljust(30)} | "
        f"{pct_str.rjust(18)} | "
        f"{str(num_outliers).rjust(18)}"
    )


Generating data quality analysis

============================= DATA QUALITY REPORT =============================
Variable Name                  | Imputation Rate (%) |  Detected Outliers
--------------------------------------------------------------------------
broadband_subscriptions        |               4.1% |                  0
capital_formation              |              20.7% |                 30
credit_to_private_sector       |              21.2% |                  5
fdi_inflows                    |               8.8% |                 40
gdp_growth                     |               2.8% |                 13
high_tech_exports              |              17.5% |                 11
ict_imports                    |              20.7% |                 10
inflation_rate                 |              12.4% |                 33
internet_usage_rate            |               4.6% |                  1
labor_productivity             |              18.4% |                  1
market

In [33]:
# Save the completed datasets
output_path = "../data/worldbank_imputed_data.csv"
outlier_log_path = "../data/worldbank_outliers_data.csv"

df_snapshot.to_csv(output_path, index=False)
df_outliers_log.to_csv(outlier_log_path, index=False)

print(f"\nClean dataset saved to '{output_path}'")
print(f"Outlier log saved to '{outlier_log_path}'")


Clean dataset saved to '../data/worldbank_imputed_data.csv'
Outlier log saved to '../data/worldbank_outliers_data.csv'


### 3.2. Data Imputation and Quality Assessment 

To meet OECD Step 3 requirements, missing values were imputed after converting the dataset into a cross-sectional format using the most recent available observation. Remaining gaps were filled using World Bank Income Group averages.

Indicators are grouped by imputation intensity:

 - **Low imputation**:
`secure_servers_density` (0.9%), `gdp_growth` (2.8%), `broadband_subscriptions` (4.1%), `internet_usage_rate` (4.6%) - these are largely based on observed data.
- **Moderate imputation**:
Most economic and development indicators (8.8% - 23.0%) - showing partial reliance on peer-group estimation.
- **High imputation**:
`real_interest_rate` (38.2%), `rd_expenditure` (41.5%), `market_capitalisation` (59.4%), `stock_market_liquidity` (61.8%) - reflecting structural data gaps in capital market reporting across countries.


### 3.3. Outlier Profile and Analysis

Outliers were identified using the IQR method ($1.5 \times \text{IQR}$).

#### Summary of Outlier Distribution:
- **Most affected variables**:
`fdi_inflows` (40), `secure_servers_density` (37), `inflation_rate` (33), `capital_formation` (30) - reflecting strong cross-country economic divergence.
- **Minimal outliers**:
`broadband_subscriptions` (0), `internet_usage_rate` (1), `labor_productivity` (1), `tertiary_enrollment` (1) - indicating stable global distributions.

Outliers are retained, as they represent structural differences rather than data errors, and will be handled during normalization.